# 06 — Model Evaluation

**Purpose:** Rigorous evaluation of all trained models on the held-out test set.

Metrics:
- **C-index** — discrimination (higher = better, >0.7 is good)
- **Integrated Brier Score (IBS)** — calibration (lower = better, <0.25 is good)
- **Time-dependent Brier score** — see calibration evolution over time

Visualisations:
- Kaplan-Meier curves (overall + by risk group)
- Predicted survival curves (sample)
- Brier score over time
- Model comparison table

---
**Inputs:** trained models, `data/processed/X_test.parquet`, `y_test.parquet`  
**Outputs:** `reports/model_comparison.csv`, `experiments/<run_name>.json`

In [ ]:
import sys, json, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_utils import load_data, load_config, DATA_PROCESSED, MODELS_DIR, REPORTS_DIR
from src.survival_models import load_model
from src.evaluation import (
    evaluate_all_models, print_comparison_table,
    plot_kaplan_meier, plot_survival_curves_comparison,
    plot_brier_score_over_time, plot_feature_importance,
    compute_brier_scores,
)
from src.experiment_tracker import log_experiment

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 40)
print('Ready.')

In [ ]:
cfg = load_config()
DURATION_COL = cfg['target']['duration_col']
EVENT_COL    = cfg['target']['event_col']

X_train = load_data('X_train', stage='processed')
X_test  = load_data('X_test',  stage='processed')
y_train = load_data('y_train', stage='processed')
y_test  = load_data('y_test',  stage='processed')

dur_train, ev_train = y_train[DURATION_COL], y_train[EVENT_COL]
dur_test,  ev_test  = y_test[DURATION_COL],  y_test[EVENT_COL]

# Load selected features
sel_path = DATA_PROCESSED / 'selected_features.json'
if sel_path.exists():
    SELECTED = json.load(open(sel_path))['selected_features']
else:
    SELECTED = list(X_train.columns)

X_train_sel = X_train[SELECTED]
X_test_sel  = X_test[SELECTED]

# Evaluation time grid
EVAL_TIMES = np.percentile(dur_test[ev_test == 1], np.linspace(5, 95, 30))

print(f'Test shape: {X_test_sel.shape}')
print(f'Eval time range: {EVAL_TIMES[0]:.1f} – {EVAL_TIMES[-1]:.1f} months')

In [ ]:
# Load all models
model_names = ['cox_ph', 'rsf', 'gbs']
models = {}
for name in model_names:
    try:
        models[name] = load_model(name)
        print(f'✓ Loaded {name}')
    except FileNotFoundError as e:
        print(f'✗ {e}')

print(f'\nModels loaded: {list(models.keys())}')

## 1. Kaplan-Meier Baseline

In [ ]:
plot_kaplan_meier(dur_test, ev_test, title='Overall Survival — Test Set (Kaplan-Meier)')

In [ ]:
# KM stratified by contract type (if available)
if 'contract_type' in X_test_sel.columns:
    plot_kaplan_meier(
        dur_test, ev_test,
        group=X_test_sel['contract_type'].map({0: 'Monthly', 1: 'One year', 2: 'Two year'}),
        title='Survival by Contract Type'
    )

## 2. C-index and Integrated Brier Score

In [ ]:
results_df = evaluate_all_models(
    models,
    X_train_sel, X_test_sel,
    dur_train, ev_train,
    dur_test, ev_test,
    EVAL_TIMES,
)

print_comparison_table(results_df)
display(results_df)

## 3. Brier Score Over Time

In [ ]:
from sksurv.util import Surv
from sksurv.metrics import brier_score

train_y = Surv.from_arrays(ev_train.astype(bool).values, dur_train.values)
test_y  = Surv.from_arrays(ev_test.astype(bool).values,  dur_test.values)

t_min = EVAL_TIMES[0]
t_max = min(dur_test.max() * 0.98, EVAL_TIMES[-1])
valid_times = EVAL_TIMES[(EVAL_TIMES >= t_min) & (EVAL_TIMES <= t_max)]

brier_dict = {}
for name, model in models.items():
    try:
        surv = model.predict_survival(X_test_sel, valid_times)
        if hasattr(surv, 'values'):
            surv = surv.values.T
        _, scores = brier_score(train_y, test_y, surv, valid_times)
        brier_dict[name] = scores
    except Exception as e:
        print(f'Could not compute Brier for {name}: {e}')

if brier_dict:
    plot_brier_score_over_time(brier_dict, valid_times)

## 4. Predicted Survival Curves (Sample Observations)

In [ ]:
plot_survival_curves_comparison(models, X_test_sel, valid_times, n_sample=5)

## 5. Risk Group Analysis

Split test set into Low/Medium/High risk groups by median risk score.

In [ ]:
best_model_name = results_df['c_index'].idxmax()
best_model = models[best_model_name]

risk_scores = best_model.predict_risk(X_test_sel)
risk_groups = pd.cut(
    risk_scores,
    bins=[risk_scores.min() - 0.01,
          np.percentile(risk_scores, 33),
          np.percentile(risk_scores, 66),
          risk_scores.max() + 0.01],
    labels=['Low risk', 'Medium risk', 'High risk'],
)

print(f'Using best model: {best_model_name}')
print('\nRisk group distribution:')
print(pd.Series(risk_groups).value_counts())

plot_kaplan_meier(
    dur_test, ev_test, group=risk_groups,
    title=f'Survival by Risk Group ({best_model_name})'
)

## 6. Business Interpretation

In [ ]:
print('=== Business Interpretation ===')
print(f'Best model     : {best_model_name}')
print(f'C-index        : {results_df.loc[best_model_name, "c_index"]:.4f}')
print(f'IBS            : {results_df.loc[best_model_name, "ibs"]:.4f}')
print()
print('Interpretation guide:')
print('  C-index 0.5  → random (coin flip)')
print('  C-index 0.7  → good discrimination')
print('  C-index 0.85 → excellent (typical clinical benchmark)')
print()
print('  IBS < 0.25   → better than predicting mean survival')
print('  IBS ~ 0.10   → well-calibrated model')

if 'cox_ph' in models:
    print('\nTop risk drivers (Cox HR > 1 = increases risk):')
    cox_summary = models['cox_ph'].summary()
    top_risk = cox_summary['exp(coef)'].sort_values(ascending=False).head(5)
    for feat, hr in top_risk.items():
        direction = '↑ RISK' if hr > 1 else '↓ risk'
        print(f'  {feat:<30}: HR = {hr:.3f}  {direction}')

## 7. Save Results & Log Experiment

In [ ]:
# Save comparison CSV
csv_path = REPORTS_DIR / 'model_comparison.csv'
results_df.to_csv(csv_path)
print(f'Model comparison saved → {csv_path}')

# Log each model as an experiment
for model_name, row in results_df.iterrows():
    log_experiment(
        run_name=f'v1_{model_name}',
        features_used=SELECTED,
        metrics={'c_index': row['c_index'], 'ibs': row['ibs']},
        params=cfg.get('models', {}).get(model_name.replace('cox_ph','cox_ph'), {}),
        notes='Initial baseline run',
    )

print('\n✓ Done. Proceed to 07_experiments_tracking.ipynb')